<!-- GENERATED by scripts/gen_notebooks.py from _variables.yml. Do not edit this cell. -->
# Module 4 · Monetary value and dollar CLV

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/marketing-statistics-workshop/blob/main/labs/python/04-monetary-clv.ipynb)

**Statistics for Marketing** · Day 2 · Python · 55-minute lab

Add a Gamma-Gamma spend model after checking that spend is independent of purchase frequency, compute discounted CLV with its uncertainty, test how it moves with horizon and discount rate, and handle the contractual case with the shifted-beta-geometric model.

**By the end of this notebook you can:**

- Check the Gamma-Gamma independence assumption with the correlation between frequency and mean spend among repeat buyers.
- Fit Gamma-Gamma and check that the true spend parameters (p, q, v) fall inside the 94% HDI.
- Compute discounted CLV with PyMC-Marketing, then write your own version that carries the uncertainty of both the purchase and the spend models.
- Show how CLV changes with the horizon in months and the monthly discount rate.
- Fit the shifted-beta-geometric (sBG) model to the Fader–Hardie retention data and project retention for later years.

**Stack:** Python 3.13, pymc-marketing, nutpie, matplotlib

**The decision:** Report what an average customer and the whole customer base are worth over 12 and 36 months with a 94% interval, and name the assumption (horizon, discount rate or spend model) that moves the number most.

**How this notebook works.** Run the cells in order (Runtime > Run all is fine: it stops at the first exercise you have not written). Each exercise has a stub to complete, a folded solution, and a checkpoint that says whether your code passed.

In [ ]:
# @title Install and check the runtime { display-mode: "form" }
# Generated by scripts/gen_notebooks.py from _variables.yml: do not edit this cell.
# On Colab or SageMaker: installs the workshop package `mktstats` at the pinned ref, and any
# package below whose version differs from the tested pins. Anywhere else: installs nothing.
import os as _os
import sys as _sys
import time as _time

_t0 = _time.time()
MKTSTATS_REF = "main"
_REPO = "https://github.com/project-delphi/marketing-statistics-workshop"
_PINS = {'matplotlib': '3.10.0', 'nutpie': '0.16.11', 'pymc-marketing': '1.2.0'}
_CONSTRAINTS = "https://raw.githubusercontent.com/project-delphi/marketing-statistics-workshop/main/environment/requirements.txt"


def _mktstats_setup():
    import importlib
    import importlib.metadata as md
    import json
    import subprocess

    root = _os.environ.get("MKTSTATS_REPO_ROOT")
    if not root:  # a notebook opened inside a checkout of the repository
        here = _os.path.abspath(_os.getcwd())
        for _ in range(4):
            if _os.path.exists(_os.path.join(here, "src", "mktstats", "__init__.py")):
                root = here
                break
            here = _os.path.dirname(here)
    if root:  # local or CI: use the checkout, install nothing
        src = _os.path.join(root, "src")
        if src not in _sys.path:
            _sys.path.insert(0, src)
        return
    hosted = (
        bool(_os.environ.get("COLAB_RELEASE_TAG"))
        or "google.colab" in _sys.modules
        or bool(_os.environ.get("SAGEMAKER_APP_TYPE") or _os.environ.get("SAGEMAKER_SPACE_NAME"))
        or _os.path.exists("/opt/ml/metadata/resource-metadata.json")
    )
    if not hosted:
        return
    try:
        info = json.loads(md.distribution("mktstats").read_text("direct_url.json") or "{}")
        have = info.get("vcs_info", {}).get("commit_id")
    except md.PackageNotFoundError:
        have = None
    want = MKTSTATS_REF if len(MKTSTATS_REF) == 40 else None
    if want is None:
        try:
            out = subprocess.run(
                ["git", "ls-remote", _REPO, MKTSTATS_REF, MKTSTATS_REF + "^{}"],
                capture_output=True, text=True, timeout=30,
            )
            lines = [x.split() for x in out.stdout.splitlines() if x.strip()]
            peeled = [x[0] for x in lines if x[1].endswith("^{}")]
            want = (peeled or [x[0] for x in lines] or [None])[0]
        except Exception:
            want = None
    if have is not None and (want is None or have == want):
        return
    print(f"Installing mktstats from {_REPO} at {MKTSTATS_REF} ...")
    done = subprocess.run(
        [_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall",
         f"git+{_REPO}@{MKTSTATS_REF}"],
        capture_output=True, text=True,
    )
    if done.returncode != 0:
        tail = "\n".join((done.stdout + done.stderr).strip().splitlines()[-10:])
        raise RuntimeError(f"Could not install mktstats:\n{tail}\nCheck the network and rerun.")
    for name in [m for m in _sys.modules if m == "mktstats" or m.startswith("mktstats.")]:
        del _sys.modules[name]
    importlib.invalidate_caches()


_mktstats_setup()
try:
    from mktstats import runtime as _runtime
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        "mktstats is not installed. In a checkout of the repository run `pip install -e .`,"
        " or set MKTSTATS_REPO_ROOT to the checkout."
    ) from _e
_runtime.ensure(_PINS, constraints_url=_CONSTRAINTS)
_os.environ["MKTSTATS_INSTALL_SECONDS"] = f"{_time.time() - _t0:.1f}"
_runtime.report()

In [ ]:
# @title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py from _variables.yml: do not edit this cell.
# WORKED_EXAMPLE runs every reference solution in place of your code: a
# complete worked example, not a sign that you did the exercises. QUICK shortens
# long steps (sampling, data size).
WORKED_EXAMPLE = False  # @param {type:"boolean"}
QUICK = False  # @param {type:"boolean"}

from mktstats import checks
from mktstats.harness import start as _start

workshop = _start(
    notebook='python/04-monetary-clv',
    content_sha='0cbabfe415ab6ce9',
    deps_sha='2a7a736b2e1872b1',
    ref='main',
    exercises={1: ('spend_frequency_correlation',), 2: ('spend_recovery',), 3: ('discounted_clv',), 4: ('clv_grid',), 5: ('counts_to_customers',)},
)

# Part A · The spend model

Modules 2 and 3 forecast how many purchases each customer will make. This lab adds money.
**Customer lifetime value (CLV)** here is the present value of what a customer will spend over a
stated horizon: for each future month k, expected purchases in that month times expected spend
per purchase, divided by (1 + d)<sup>k</sup>, where d is the **monthly discount rate**. It is
revenue, not margin; Module 5 applies the margin.

The data is the workshop's synthetic retailer: 5,000 customers acquired in the first half of
2024, with purchases generated by a Pareto/NBD process and spend by a Gamma-Gamma process whose
true parameters are known, so every estimate can be checked against the truth. The
**calibration period** (the history the models see) ends on 2024-12-29. CDNOW, a real online CD
retailer, is the benchmark with published estimates.

The RFM table has one row per customer: `frequency` = repeat purchases (purchase days after
the first), `recency` = weeks from the first to the last purchase, `T` = weeks from the first
purchase to the end of calibration, and `monetary_value` = mean spend over the repeat
purchases (0 for a customer with none).

**QUICK.** With QUICK on (the harness cell), every sampler draws 2 chains × 300 instead of
2 × 1,000. Intervals are then wider and may not match the module page, and a decision near its
threshold can flip. That is a lesson about sample size, not a bug.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from pymc_extras.prior import Prior
from pymc_marketing import clv
from scipy.special import betaln

from mktstats import data, recovery, synth

SEED = 2026                     # the retailer's seed: the committed data and truth.json use it
FIT_SEED = 404                  # the seed of every sampler in this lab
CHAINS = 2
DRAWS = 300 if QUICK else 1000  # posterior draws per chain, and as many tuning steps
THIN = 2 if QUICK else 4        # the CLV arrays keep every THIN-th draw (300 or 500 in all)
INTERVAL = 0.94                 # every interval here is a 94% highest-density interval (HDI)
WEEKS_PER_MONTH = 30.4375 / 7   # PyMC-Marketing's days per month, in weeks
MONTHLY_RATE = 0.008            # 0.8% a month, about 10% a year (the truth uses 10% a year)

retailer = synth.retailer(seed=SEED)
truth = retailer.truth
tx, customers = retailer.transactions, retailer.customers
rfm = data.retailer_rfm(tx, customers, truth)

cdnow_tx = data.load_cdnow()
cdnow_rfm = data.rfm_summary(cdnow_tx, "id", "date", "spent", observation_period_end="1997-09-30",
                             time_unit="D", time_scaler=7)

print(f"Retailer: {len(rfm):,} customers; calibration ends {truth['calibration_end']};"
      f" {(rfm['frequency'] > 0).sum():,} made at least one repeat purchase.")
print(f"CDNOW: {len(cdnow_rfm):,} customers; calibration ends 1997-09-30;"
      f" {(cdnow_rfm['frequency'] > 0).sum():,} made at least one repeat purchase.")
rfm.head()

## Exercise 1 · Is spend independent of frequency? (6 minutes)

The spend model you fit next, the **Gamma-Gamma model**, assumes that how much a customer
spends per purchase does not depend on how often they buy. Check that before fitting anything.

**Predict.** Among repeat buyers, will the correlation between the number of repeat purchases
and the mean spend per purchase be below 0.2, between 0.2 and 0.5, or above 0.5? Write one
guess for the retailer and one for CDNOW.

**Task.** Write `spend_frequency_correlation(rfm)` returning, as a float, the Pearson
correlation between `frequency` and `monetary_value` among repeat buyers (`frequency > 0`).

In [ ]:
def spend_frequency_correlation(rfm):
    # TODO 1: keep the repeat buyers, then correlate frequency with monetary_value
    raise NotImplementedError("TODO 1")

In [ ]:
# @title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def spend_frequency_correlation(rfm):
    repeat = rfm[rfm["frequency"] > 0]
    return float(repeat["frequency"].corr(repeat["monetary_value"]))

In [ ]:
with workshop.checkpoint(1):
    corr_retailer = spend_frequency_correlation(rfm)
    corr_cdnow = spend_frequency_correlation(cdnow_rfm)
    for name, got in (("the retailer", corr_retailer), ("CDNOW", corr_cdnow)):
        assert isinstance(got, float), (
            f"Checkpoint 1: for {name} your function returned a {type(got).__name__}, not a float."
            " Return one number, for example float(series_a.corr(series_b)). To move on, run"
            " workshop.use_reference(1)."
        )
    repeat = rfm[rfm["frequency"] > 0]
    expected = float(np.corrcoef(repeat["frequency"], repeat["monetary_value"])[0, 1])
    everyone = float(np.corrcoef(rfm["frequency"], rfm["monetary_value"])[0, 1])
    cause = (
        "You kept the customers with no repeat purchase: their monetary_value is 0, which"
        " creates a correlation that is not in the spend data."
        if abs(corr_retailer - everyone) < 1e-6
        else "Check that you correlate frequency with monetary_value (not recency or T)."
    )
    assert abs(corr_retailer - expected) < 1e-9, (
        f"Checkpoint 1: your correlation on the retailer is {corr_retailer:.4f}; among repeat"
        f" buyers it is {expected:.4f}. {cause} Keep the rows with frequency > 0 before"
        " correlating. To move on, run workshop.use_reference(1)."
    )
    # Published check: 946 CDNOW repeat buyers, correlation 0.114 (Fader & Hardie, note 025).
    assert abs(corr_cdnow - 0.1139) < 5e-4, (
        f"Checkpoint 1: your CDNOW correlation is {corr_cdnow:.4f}; on the 946 repeat buyers it"
        f" is 0.114. {cause} To move on, run workshop.use_reference(1)."
    )
    print(f"Correlation among repeat buyers: retailer {corr_retailer:.3f}"
          f" ({len(repeat):,} customers), CDNOW {corr_cdnow:.3f}"
          f" ({int((cdnow_rfm['frequency'] > 0).sum()):,} customers).")

**Explain.** Compare with your guesses. Which of the two numbers would make you more cautious
about the Gamma-Gamma model, and why does a small correlation not prove that spend and
frequency are independent?

<details><summary>Why this solution works</summary>

Pearson's correlation measures a straight-line association. A customer with no repeat purchase
has no observed repeat spend (`monetary_value` is set to 0), so including them manufactures a
correlation: they have both the lowest frequency and a spend of zero. The retailer's spend was
generated independently of its purchases, so its correlation is close to 0; CDNOW's 0.11 is
small, which Fader and Hardie (note 025) accept. A small correlation is consistent with
independence but does not prove it: a dependence that is not a straight line, or one confined to
a few heavy buyers, can hide behind it. If the correlation were large, you would need a model of
spend and frequency together rather than two separate models.
</details>

### Run · Fit the spend model

The Gamma-Gamma model (Fader & Hardie, note 025) says each transaction of customer i has value
z ~ Gamma(p, ν<sub>i</sub>), and the ν<sub>i</sub> vary across customers as Gamma(q, v). So a
customer's mean spend per purchase is p / ν<sub>i</sub>, and the population mean is
p·v / (q − 1). It is fitted on repeat buyers only: a customer's first purchase is not part of
`monetary_value`, and the model needs a spend above 0.

**Priors.** PyMC-Marketing's default priors (Weibull(2, 1) for p and q) pull p well below a
true value of 6 on data this size: over 30 seeds the MAP estimate of p averaged 4.6 with the
defaults and 5.8 with weaker priors (`mktstats.recovery`). This lab uses those weaker priors,
`recovery.WEAK_GAMMA_GAMMA_PRIORS`: HalfNormal(10) for p and q, HalfNormal(100) for v.

The fit draws from the posterior with MCMC (the nutpie sampler). With QUICK on it draws
2 × 300; expect a few seconds on a laptop, longer on Colab, where the first fit of each model
also compiles it.

In [ ]:
gg_priors = {name: Prior(dist, **kw) for name, (dist, kw) in recovery.WEAK_GAMMA_GAMMA_PRIORS.items()}
repeat_buyers = rfm.loc[rfm["frequency"] > 0, ["customer_id", "frequency", "monetary_value"]]

t0 = time.time()
gg = clv.GammaGammaModel(model_config=gg_priors)
gg.fit(data=repeat_buyers, method="mcmc", nuts_sampler="nutpie", chains=CHAINS, draws=DRAWS,
       tune=DRAWS, random_seed=FIT_SEED, progressbar=False)
gg_idata = gg.idata
divergences = int(gg_idata.sample_stats["diverging"].sum())
print(f"Gamma-Gamma: {CHAINS} chains x {DRAWS} draws in {time.time() - t0:.1f} s,"
      f" {divergences} divergent transitions, on {len(repeat_buyers):,} repeat buyers.")

## Exercise 2 · Recover the spend parameters (7 minutes)

**Predict.** A customer made one repeat purchase of $200; the population mean is about $30. Will
the model's expected spend for this customer be closer to $200 or to $30?

**Task.** Write `spend_recovery(gg_idata, truth, prob=0.94)` returning a DataFrame with one row
per parameter (`p`, `q`, `v`) and columns `param`, `truth`, `mean` (posterior mean),
`hdi_low`, `hdi_high` (the `prob` HDI) and `covered` (is the truth inside the HDI?). The
posterior draws are `gg_idata.posterior["p"]` and so on; `truth` is a dict like
`{"p": 6.0, "q": 4.0, "v": 15.0}`. Compute the HDI with `checks.interval(draws, prob, "hdi")`,
never with a library default (ArviZ 1.x defaults to an 89% equal-tailed interval).

In [ ]:
def spend_recovery(gg_idata, truth, prob=0.94):
    # TODO 2: one row per parameter: param, truth, mean, hdi_low, hdi_high, covered
    raise NotImplementedError("TODO 2")

In [ ]:
# @title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def spend_recovery(gg_idata, truth, prob=0.94):
    rows = []
    for name in ("p", "q", "v"):
        draws = gg_idata.posterior[name]
        lo, hi = checks.interval(draws, prob, "hdi")
        rows.append({"param": name, "truth": truth[name], "mean": float(draws.mean()),
                     "hdi_low": lo, "hdi_high": hi, "covered": lo <= truth[name] <= hi})
    return pd.DataFrame(rows)

In [ ]:
with workshop.checkpoint(2):
    recovery_table = spend_recovery(gg_idata, truth["gamma_gamma"], prob=INTERVAL)
    checks.columns(recovery_table, ["param", "truth", "mean", "hdi_low", "hdi_high", "covered"],
                   name="spend_recovery(...)")
    assert sorted(recovery_table["param"]) == ["p", "q", "v"], (
        f"Checkpoint 2: expected one row each for p, q and v; got {list(recovery_table['param'])}."
        " To move on, run workshop.use_reference(2)."
    )
    for row in recovery_table.itertuples():
        draws = gg_idata.posterior[row.param]
        lo, hi = checks.interval(draws, INTERVAL, "hdi")
        width = hi - lo
        assert abs(row.mean - float(draws.mean())) <= 1e-9 * abs(row.mean), (
            f"Checkpoint 2: your mean of {row.param} is {row.mean:.4g}; the posterior mean is"
            f" {float(draws.mean()):.4g}. Average over all chains and draws. To move on, run"
            " workshop.use_reference(2)."
        )
        assert abs(row.hdi_low - lo) <= 0.01 * width and abs(row.hdi_high - hi) <= 0.01 * width, (
            f"Checkpoint 2: your interval for {row.param} is [{row.hdi_low:.4g},"
            f" {row.hdi_high:.4g}]; the 94% HDI is [{lo:.4g}, {hi:.4g}]. A library default (an"
            " 89% interval, or an equal-tailed one) is the usual cause. Use"
            " checks.interval(draws, prob, 'hdi'). To move on, run workshop.use_reference(2)."
        )
        assert bool(row.covered) == (lo <= truth["gamma_gamma"][row.param] <= hi), (
            f"Checkpoint 2: `covered` for {row.param} is {row.covered}, but the truth"
            f" {truth['gamma_gamma'][row.param]} is {'inside' if lo <= truth['gamma_gamma'][row.param] <= hi else 'outside'}"
            f" [{lo:.4g}, {hi:.4g}]. To move on, run workshop.use_reference(2)."
        )
    # p, q and v are strongly correlated in the posterior, so one miss can happen by chance.
    checks.k_of_K_in_interval(truth["gamma_gamma"],
                              {k: gg_idata.posterior[k] for k in ("p", "q", "v")}, k=2,
                              prob=INTERVAL, kind="hdi", name="the Gamma-Gamma parameters")
recovery_table

Now the customer from the prediction: one repeat purchase of $200.

In [ ]:
one_purchase = pd.DataFrame({"customer_id": [0], "frequency": [1], "monetary_value": [200.0]})
expected_200 = gg.expected_customer_spend(data=one_purchase)
population = gg_idata.posterior["p"] * gg_idata.posterior["v"] / (gg_idata.posterior["q"] - 1)
lo, hi = checks.interval(expected_200, INTERVAL, "hdi")
print(f"Expected spend of the $200 customer: ${float(expected_200.mean()):.0f}"
      f" (94% HDI ${lo:.0f} to ${hi:.0f}); population mean ${float(population.mean()):.2f}.")

**Explain.** Was your guess right? Look at the formula below and say which number in it grows
when a customer has made more purchases, and what that does to their expected spend.

<details><summary>Why this solution works</summary>

The expected spend is a weighted average,
E[spend<sub>i</sub>] = (1 − w<sub>i</sub>) · p·v/(q − 1) + w<sub>i</sub> · z̄<sub>i</sub> with
w<sub>i</sub> = p·x<sub>i</sub> / (p·x<sub>i</sub> + q − 1), where z̄<sub>i</sub> is the customer's
mean spend over x<sub>i</sub> repeat purchases. With p near 6 and q near 4, one purchase already
gets a weight of about 6/9 = 0.67, so the $200 customer lands nearer $200 than $30; with more
purchases the weight approaches 1. A smaller p (more variable transactions) would pull the
customer further toward the population mean. The recovery table also shows why the priors matter:
p, q and v trade off against each other, so a prior that pulls p down moves v as well while the
population mean barely changes.
</details>

### Run · The published benchmark on CDNOW

Fader and Hardie (note 025) fit the same model to CDNOW's 946 repeat buyers by maximum
likelihood and report p = 6.25, q = 3.74, v = 15.44. A MAP fit with flat priors (`HalfFlat`, no
preference for any positive value) is maximum likelihood, so the library should reproduce them.

In [ ]:
PUBLISHED_GG = {"p": 6.25, "q": 3.74, "v": 15.44}
flat = {name: Prior("HalfFlat") for name in PUBLISHED_GG}
gg_cdnow = clv.GammaGammaModel(model_config=flat)
gg_cdnow.fit(data=cdnow_rfm.loc[cdnow_rfm["frequency"] > 0,
                                ["customer_id", "frequency", "monetary_value"]],
             method="map", progressbar=False)
cdnow_estimates = gg_cdnow.fit_summary()
pd.DataFrame({"published": PUBLISHED_GG,
              "this fit": {k: float(cdnow_estimates[k]) for k in PUBLISHED_GG}}).round(3)

In [ ]:
with workshop.checkpoint(label="CDNOW Gamma-Gamma benchmark"):
    for name, value in PUBLISHED_GG.items():
        got = float(cdnow_estimates[name])
        assert abs(got / value - 1) <= 0.01, (
            f"The CDNOW fit gives {name} = {got:.3f}; note 025 reports {value} (allowed 1%)."
            " This is provided code: rerun the cell above, and tell the instructor if it fails"
            " again."
        )

# Part B · Discounted CLV

CLV needs a forecast of purchases. The **transaction model** here is the BG/NBD model from
Module 2, not the Pareto/NBD that generated the retailer's data: a Pareto/NBD MCMC fit took 57 s
for 2 × 300 draws on CDNOW on a 10-core laptop and 231 s on Colab (DECISIONS.md, spike S2), too
long for this lab, and the BG/NBD forecasts similarly (Fader, Hardie & Lee 2005) while sampling in
seconds. The price is a small misspecification, which is why the CLV checkpoint below allows a
wider gap to the truth than a recovery check would. The stretch compares a Pareto/NBD fit.

PyMC-Marketing's default BG/NBD priors place the dropout parameters a and b through two others
(`phi_dropout`, `kappa_dropout`); a and b are recorded in the posterior as derived quantities.
With QUICK on the fit draws 2 × 300.

In [ ]:
t0 = time.time()
bgnbd = clv.BetaGeoModel()
with warnings.catch_warnings():
    # pymc-marketing 1.2.0 passes PyMC an argument PyMC has deprecated (merge_dataset): the
    # FutureWarning is about the library, not your code. It also prints a "Computing ..." bar.
    warnings.simplefilter("ignore", FutureWarning)
    bgnbd.fit(data=rfm[["customer_id", "frequency", "recency", "T"]], method="mcmc",
              nuts_sampler="nutpie", chains=CHAINS, draws=DRAWS, tune=DRAWS,
              random_seed=FIT_SEED, progressbar=False)
print(f"BG/NBD: {CHAINS} chains x {DRAWS} draws in {time.time() - t0:.1f} s,"
      f" {int(bgnbd.idata.sample_stats['diverging'].sum())} divergent transitions.")
pd.DataFrame(
    [{"param": k, "mean": float(bgnbd.idata.posterior[k].mean()),
      **dict(zip(("hdi_low", "hdi_high"),
                 checks.interval(bgnbd.idata.posterior[k], INTERVAL, "hdi")))}
     for k in ("r", "alpha", "a", "b")]
).round(3)

### Run · CLV with PyMC-Marketing

`GammaGammaModel.expected_customer_lifetime_value` combines the two models. Two easy mistakes:

- `future_t` is always in **months**, whatever `time_unit` is. `time_unit="W"` only says that
  the RFM table is in weeks.
- `discount_rate` is **monthly**: 0.008 is 0.8% a month, about 10% a year.

What the source shows (`clv/models/gamma_gamma.py`): the method multiplies the purchase forecast
by each customer's **posterior mean** spend. So its interval reflects only the purchase model's
uncertainty, not the spend model's. Exercise 3 fixes that.

The arrays below have one value per draw and customer. To keep them in memory on Colab, both
models keep every `THIN`-th draw (`thin_fit_result`). The truth is the sum over channels of
`truth["value_by_channel"]["existing_customers_at_cal_end"][c]["total_discounted_clv"]`: revenue
over the 52 weeks after calibration, from each customer's true purchase rate, dropout rate, mean
spend and alive status, discounted continuously at 10% a year. The library uses 12 months of
30.4375 days (52.2 weeks) and discounts each month at its end; that convention gap is well under
1%.

In [ ]:
bgnbd_clv = bgnbd.thin_fit_result(THIN)
gg_clv = gg.thin_fit_result(THIN)
rfm_clv = rfm[["customer_id", "frequency", "recency", "T", "monetary_value"]]

clv_lib = gg_clv.expected_customer_lifetime_value(
    transaction_model=bgnbd_clv, data=rfm_clv, future_t=12, discount_rate=MONTHLY_RATE,
    time_unit="W",
)
total_lib = clv_lib.sum("customer_id")
lib_lo, lib_hi = checks.interval(total_lib, INTERVAL, "hdi")
truth_total = sum(c["total_discounted_clv"]
                  for c in truth["value_by_channel"]["existing_customers_at_cal_end"].values())
print(f"clv_lib dims {clv_lib.dims}, shape {clv_lib.shape}")
print(f"Total 12-month CLV, library: ${float(total_lib.mean()):,.0f}"
      f" (94% HDI ${lib_lo:,.0f} to ${lib_hi:,.0f}); truth ${truth_total:,.0f}.")

### Run · Purchases in each month, and spend draws

The next cell computes, for every kept draw and customer, the expected purchases **in** each
month k = 1, …, 36: the cumulative forecast at k months minus the forecast at k − 1 months, with
months of 30.4375 days as in the library. It also takes the spend draws, one expected spend per
draw and customer, from `gg_clv.expected_customer_spend`. It is the slowest cell of the lab
(about 20 seconds on a laptop with FULL settings; longer on Colab): read Exercise 3 while it runs.

In [ ]:
def purchases_in_month(model, rfm, months):
    """Expected purchases of each customer in month k = 1..months (not cumulative), per draw."""
    steps = np.arange(1, months + 1)
    cumulative = xr.concat(
        [model.expected_purchases(data=rfm, future_t=k * WEEKS_PER_MONTH) for k in steps],
        dim=xr.DataArray(steps, dims="month", name="month"),
    )
    in_month = cumulative - cumulative.shift(month=1, fill_value=0.0)
    return in_month.transpose("chain", "draw", "customer_id", "month")


t0 = time.time()
purchases_by_month = purchases_in_month(bgnbd_clv, rfm_clv, 36)
spend = gg_clv.expected_customer_spend(data=rfm_clv)
print(f"purchases_by_month {purchases_by_month.dims} {purchases_by_month.shape},"
      f" spend {spend.dims} {spend.shape}, in {time.time() - t0:.1f} s")

## Exercise 3 · Your own CLV that carries both uncertainties (10 minutes)

Pair each draw of the purchase model with a draw of the spend model, and the interval carries
both uncertainties:

CLV<sub>i</sub> = Σ<sub>k=1..H</sub> s<sub>i</sub> · n<sub>ik</sub> / (1 + d)<sup>k</sup>,

where s<sub>i</sub> is customer i's expected spend per purchase (one value per draw),
n<sub>ik</sub> the expected purchases in month k, d the monthly discount rate and H the horizon in
months.

**Predict.** The library's 94% HDI for the total 12-month CLV is printed above. With spend
uncertainty added, will the interval for the **total** be about the same width, up to twice as
wide, or more than twice as wide?

**Task.** Write `discounted_clv(purchases_by_month, spend, monthly_rate)` returning an
`xr.DataArray` with dims `(chain, draw, customer_id)`. `purchases_by_month` has dims
`(chain, draw, customer_id, month)` with months numbered 1, 2, …; `spend` has dims
`(chain, draw, customer_id)`. Use xarray arithmetic, which lines arrays up by dimension name, and
the `month` coordinate for k.

In [ ]:
def discounted_clv(purchases_by_month, spend, monthly_rate):
    # TODO 3: discount each month's purchases, sum over months, multiply by spend
    raise NotImplementedError("TODO 3")

In [ ]:
# @title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def discounted_clv(purchases_by_month, spend, monthly_rate):
    discount = (1 + monthly_rate) ** -purchases_by_month["month"]
    return (purchases_by_month * discount).sum("month") * spend

In [ ]:
with workshop.checkpoint(3):
    first_year = purchases_by_month.sel(month=slice(1, 12))
    # (1) With spend fixed at its posterior mean, it must match the library.
    clv_mean_spend = discounted_clv(first_year, spend.mean(("chain", "draw")), MONTHLY_RATE)
    assert isinstance(clv_mean_spend, xr.DataArray) and set(clv_mean_spend.dims) == {
        "chain", "draw", "customer_id"}, (
        f"Checkpoint 3: expected an xr.DataArray with dims (chain, draw, customer_id), got"
        f" {getattr(clv_mean_spend, 'dims', type(clv_mean_spend).__name__)}. Sum over 'month'"
        " and keep the other dims. To move on, run workshop.use_reference(3)."
    )
    rel = float((abs(clv_mean_spend - clv_lib) / abs(clv_lib)).max())
    assert rel <= 1e-6, (
        f"Checkpoint 3: with spend at its posterior mean, your CLV differs from the library's by"
        f" up to {rel:.2%} for a customer. The usual causes: discounting month k by"
        " (1 + d) ** (k - 1) instead of (1 + d) ** k, or cumulating purchases that are already"
        " per month. Check the exponent and that you sum purchases_by_month over 'month'. To"
        " move on, run workshop.use_reference(3)."
    )
    # (2) With spend draws, per-customer intervals must widen (94% equal-tailed, per customer).
    clv_12 = discounted_clv(first_year, spend, MONTHLY_RATE).transpose("chain", "draw", "customer_id")

    def eti_width(x):
        q = x.quantile([0.03, 0.97], dim=("chain", "draw"))
        return q.sel(quantile=0.97) - q.sel(quantile=0.03)

    widening = float((eti_width(clv_12) / eti_width(clv_lib)).median())
    assert widening > 1.01, (
        f"Checkpoint 3: with spend draws, the median customer's 94% interval is {widening:.3f}"
        " times the library's; it should be wider. The usual cause: averaging spend over draws"
        " inside your function. Multiply by `spend` as given, draw by draw. To move on, run"
        " workshop.use_reference(3)."
    )
    # (3) The total against the truth. Tolerance 12%: the reference run is 7% above the truth in
    # QUICK and FULL (BG/NBD fitted to Pareto/NBD data); the 12-month vs 52-week and monthly vs
    # continuous discounting conventions differ by well under 1%.
    total_12 = clv_12.sum("customer_id")
    gap = float(total_12.mean()) / truth_total - 1
    assert abs(gap) <= 0.12, (
        f"Checkpoint 3: your total 12-month CLV is ${float(total_12.mean()):,.0f}; the truth is"
        f" ${truth_total:,.0f} ({gap:+.1%}, allowed 12%). Check the discount rate (monthly,"
        " 0.008) and that only months 1 to 12 are summed. To move on, run"
        " workshop.use_reference(3)."
    )

own_lo, own_hi = checks.interval(total_12, INTERVAL, "hdi")
print(f"Total 12-month CLV, library (mean spend): ${float(total_lib.mean()):,.0f}"
      f"  94% HDI ${lib_lo:,.0f} to ${lib_hi:,.0f}")
print(f"Total 12-month CLV, with spend draws:     ${float(total_12.mean()):,.0f}"
      f"  94% HDI ${own_lo:,.0f} to ${own_hi:,.0f}")
print(f"Width of the total's interval, yours / library's: {(own_hi - own_lo) / (lib_hi - lib_lo):.2f}")
print(f"Width of the median customer's interval, yours / library's: {widening:.2f}")

**Explain.** Compare with your prediction. Spend uncertainty barely widens the interval of the
total, yet it clearly widens a single customer's. Which customers carry most of the total, and
how much of their expected spend comes from their own purchases rather than from p, q and v?

<details><summary>Why this solution works</summary>

Pairing draw j of the purchase posterior with draw j of the spend posterior gives draws from the
joint posterior only if the two posteriors are independent, which is the Gamma-Gamma independence
assumption from Exercise 1 again. The total is dominated by frequent buyers, whose expected spend
is mostly their own observed mean (weight close to 1), which is data rather than a parameter;
and the population mean spend is pinned down by over 3,000 repeat buyers. So the purchase model's
uncertainty dominates the total. For a single customer with one or two purchases, the weight on
the population parameters is larger and the spend uncertainty shows. Both intervals are for an
*expected* value: most of the uncertainty about what one customer will actually spend is that
customer's own behaviour, which neither interval includes.
</details>

## Exercise 4 · Horizon and discount-rate sensitivity (7 minutes)

`clv_fn(months, rate)` below is your `discounted_clv` with its inputs bound: it returns the
per-customer CLV draws for a horizon of `months` (up to 36) and a monthly rate.

**Predict.** Which moves the total CLV more: going from 12 to 36 months at 0.8% a month, or going
from 0% to 2% a month at 36 months? Pick one.

**Task.** Write `clv_grid(clv_fn, horizons=(12, 24, 36), rates=(0.0, 0.008, 0.02))` returning a
DataFrame with one row per (horizon, rate) and columns `horizon_months`, `monthly_rate`,
`total_mean` (posterior mean of the total over customers), `hdi_low`, `hdi_high` (its 94% HDI).

In [ ]:
def clv_fn(months, rate):
    """Per-customer CLV draws for a horizon of `months` (1 to 36) at a monthly `rate`."""
    return discounted_clv(purchases_by_month.sel(month=slice(1, months)), spend, rate)

In [ ]:
def clv_grid(clv_fn, horizons=(12, 24, 36), rates=(0.0, 0.008, 0.02)):
    # TODO 4: for each horizon and rate, total over customers, then mean and 94% HDI
    raise NotImplementedError("TODO 4")

In [ ]:
# @title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
def clv_grid(clv_fn, horizons=(12, 24, 36), rates=(0.0, 0.008, 0.02)):
    rows = []
    for months in horizons:
        for rate in rates:
            total = clv_fn(months, rate).sum("customer_id")
            lo, hi = checks.interval(total, 0.94, "hdi")
            rows.append({"horizon_months": months, "monthly_rate": rate,
                         "total_mean": float(total.mean()), "hdi_low": lo, "hdi_high": hi})
    return pd.DataFrame(rows)

In [ ]:
with workshop.checkpoint(4):
    grid = clv_grid(clv_fn)
    checks.columns(grid, ["horizon_months", "monthly_rate", "total_mean", "hdi_low", "hdi_high"],
                   name="clv_grid(clv_fn)")
    assert len(grid) == 9, (
        f"Checkpoint 4: expected 9 rows (3 horizons x 3 rates), got {len(grid)}. To move on, run"
        " workshop.use_reference(4)."
    )
    for rate, part in grid.groupby("monthly_rate"):
        checks.monotone(part.sort_values("horizon_months")["total_mean"], increasing=True,
                        name=f"Total CLV by horizon at {rate:.1%} a month")
    for months, part in grid.groupby("horizon_months"):
        checks.monotone(part.sort_values("monthly_rate")["total_mean"], increasing=False,
                        name=f"Total CLV by discount rate at {months} months")
    assert ((grid["hdi_low"] <= grid["total_mean"]) & (grid["total_mean"] <= grid["hdi_high"])).all(), (
        "Checkpoint 4: some posterior means fall outside their HDI: check the column order."
        " To move on, run workshop.use_reference(4)."
    )
    cell = grid[(grid["horizon_months"] == 12) & np.isclose(grid["monthly_rate"], MONTHLY_RATE)]
    checks.close(float(cell["total_mean"].iloc[0]), float(total_12.mean()), rel=1e-9,
                 name="the (12 months, 0.8%) cell against Exercise 3's total")

grid.assign(monthly_rate=grid["monthly_rate"].map("{:.1%}".format)).round(0)

Look at how much each line rises from 24 to 36 months compared with 12 to 24, and how far apart
the three discount rates are at 36 months. Bars are 94% HDIs.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for (rate, part), marker in zip(grid.groupby("monthly_rate"), ("o", "s", "^"), strict=True):
    part = part.sort_values("horizon_months")
    ax.errorbar(part["horizon_months"], part["total_mean"] / 1e6,
                yerr=[(part["total_mean"] - part["hdi_low"]) / 1e6,
                      (part["hdi_high"] - part["total_mean"]) / 1e6],
                marker=marker, capsize=4, label=f"{rate:.1%} a month")
ax.set_xticks([12, 24, 36])
ax.set_xlabel("Horizon (months)")
ax.set_ylabel("Total CLV ($ millions)")
ax.set_title("Total CLV of the 5,000 customers (94% HDI)")
ax.legend(title="Discount rate")
plt.tight_layout()
plt.show()

**Explain.** Compare with your pick. Why does the step from 24 to 36 months add less than the
step from 12 to 24? Point to the plot.

<details><summary>Why this solution works</summary>

0.8% a month is about 10% a year; 2% a month is about 27% a year. Longer horizons add more
months, but each later month adds less: customers keep dropping out (P(alive) decays), and later
months are discounted more. So the curve flattens. The horizon is a business choice (how long the
plan runs and how far you trust the model), and here it moves the total more than any of the
discount rates. The discount rate matters more the longer the horizon, because discounting
compounds.
</details>

# Part C · Contractual customers

Subscriptions are **contractual**: you see the moment a customer leaves, because they do not
renew. The **shifted-beta-geometric (sBG)** model says each customer has a fixed probability θ of
leaving at each renewal, and θ varies across customers as Beta(α, β). These α and β are **not**
the α and β of the Pareto/NBD or BG/NBD: they describe churn probabilities. The data are Fader and
Hardie's (2007) survivors out of 1,000 customers of a "Regular" and a "High End" segment, years
0 to 12. Fit on years 0 to 7, as the paper does, and project years 8 to 12.

In [ ]:
retention = data.load_sbg_retention()
train = retention[retention["year"] <= 7]
SBG_FLAT = {"alpha": Prior("HalfFlat", dims="cohort"), "beta": Prior("HalfFlat", dims="cohort")}
FADER_HARDIE_2007 = {"regular": {"alpha": 0.704, "beta": 1.182},
                     "highend": {"alpha": 0.668, "beta": 3.806}}
retention.set_index("year").T

## Exercise 5 · Encode the retention counts and fit the sBG (10 minutes)

`ShiftedBetaGeoModel` wants one row per customer: `recency` is the year in which the customer
left, and `T` is the same for everyone in a cohort. Its likelihood is **censored** at T (read in
the source): a row with `recency == T` means "still active at T, left at T or later".

**Predict.** Will the year-to-year retention rate of Regular customers rise, fall or stay flat
after year 7? Pick one.

**Task.** Write `counts_to_customers(alive, cohort)` where `alive` lists the survivors at years
0, 1, …, n (so `alive[0]` is the cohort size). Return a DataFrame with columns `customer_id`
(unique, e.g. `"regular-17"`), `recency`, `T` and `cohort`: the customers lost during year t
(`alive[t-1] - alive[t]` of them) get `recency = t`; the customers still active after year n get
`recency = T`; and every row has **`T = n + 1`**.

In [ ]:
def counts_to_customers(alive, cohort):
    # TODO 5: one row per customer; lost in year t -> recency t; survivors -> recency T = n + 1
    raise NotImplementedError("TODO 5")

In [ ]:
# @title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def counts_to_customers(alive, cohort):
    n = len(alive) - 1
    recency = []
    for t in range(1, n + 1):
        recency += [t] * int(alive[t - 1] - alive[t])  # left during year t
    recency += [n + 1] * int(alive[n])  # still active after year n: censored at T = n + 1
    return pd.DataFrame({"customer_id": [f"{cohort}-{i}" for i in range(len(recency))],
                         "recency": recency, "T": n + 1, "cohort": cohort})

In [ ]:
with workshop.checkpoint(5):
    toy = counts_to_customers([10, 6, 4], "toy")
    checks.columns(toy, ["customer_id", "recency", "T", "cohort"], name="counts_to_customers(...)")
    assert len(toy) == 10 and toy["customer_id"].is_unique, (
        f"Checkpoint 5: a cohort of 10 should give 10 rows with unique customer_id; got"
        f" {len(toy)} rows. To move on, run workshop.use_reference(5)."
    )
    assert (toy["T"] == 3).all(), (
        f"Checkpoint 5: with survivors at years 0, 1, 2 (n = 2), every row needs T = n + 1 = 3;"
        f" got T = {sorted(int(x) for x in toy['T'].unique())}. Survivors after year n need"
        " recency = T = n + 1:"
        " the likelihood is censored at T, so T = n would say they might have left in year n."
        " To move on, run workshop.use_reference(5)."
    )
    counts = toy["recency"].value_counts().sort_index().to_dict()
    assert counts == {1: 4, 2: 2, 3: 4}, (
        f"Checkpoint 5: [10, 6, 4] means 4 customers left in year 1, 2 in year 2 and 4 are still"
        f" active (recency 3 = T); your recency counts are {counts}. To move on, run"
        " workshop.use_reference(5)."
    )
    sbg_data = pd.concat([counts_to_customers(train[c].tolist(), c) for c in FADER_HARDIE_2007],
                         ignore_index=True)
    sbg_map = clv.ShiftedBetaGeoModel(model_config=SBG_FLAT)
    sbg_map.fit(data=sbg_data, method="map", progressbar=False)
    sbg_estimates = sbg_map.fit_summary()
    for cohort, published in FADER_HARDIE_2007.items():
        for name, value in published.items():
            got = float(sbg_estimates[f"{name}[{cohort}]"])
            assert abs(got / value - 1) <= 0.01, (
                f"Checkpoint 5: the flat-prior MAP gives {name} = {got:.3f} for {cohort}; Fader"
                f" and Hardie (2007) report {value} (allowed 1%). Survivors after year n need"
                " recency = T = n + 1. To move on, run workshop.use_reference(5)."
            )
print(sbg_estimates.round(3).to_string())

### Run · Project retention with its uncertainty

The MAP fit reproduces the paper. For an interval on the projection, draw from the posterior
(flat priors, MCMC), then compute the survival curve S(t) = B(α, β + t) / B(α, β) for each draw,
where B is the beta function.

In [ ]:
t0 = time.time()
sbg = clv.ShiftedBetaGeoModel(model_config=SBG_FLAT)
sbg.fit(data=sbg_data, method="mcmc", nuts_sampler="nutpie", chains=CHAINS, draws=DRAWS,
        tune=DRAWS, random_seed=FIT_SEED, progressbar=False)
print(f"sBG: {CHAINS} chains x {DRAWS} draws in {time.time() - t0:.1f} s,"
      f" {int(sbg.idata.sample_stats['diverging'].sum())} divergent transitions.")

years = xr.DataArray(np.arange(13), dims="year", coords={"year": np.arange(13)})
alpha_d, beta_d = sbg.idata.posterior["alpha"], sbg.idata.posterior["beta"]
survival = np.exp(betaln(alpha_d, beta_d + years) - betaln(alpha_d, beta_d))  # (chain, draw, cohort, year)

rows = []
for cohort in FADER_HARDIE_2007:
    for year in range(8, 13):
        s = 1000 * survival.sel(cohort=cohort, year=year)
        lo, hi = checks.interval(s, INTERVAL, "hdi")
        rows.append({"cohort": cohort, "year": year,
                     "actual": int(retention.set_index("year").loc[year, cohort]),
                     "projected": float(s.mean()), "hdi_low": lo, "hdi_high": hi})
projection = pd.DataFrame(rows)
projection.round(0)

Look at whether the actual survivors for years 8 to 12 (black crosses) fall inside the shaded 94%
HDI bands, which the model drew from years 0 to 7 only.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for cohort, marker in (("regular", "o"), ("highend", "s")):
    s = 1000 * survival.sel(cohort=cohort)
    lo_hi = np.array([checks.interval(s.sel(year=y), INTERVAL, "hdi") for y in range(13)])
    ax.fill_between(range(13), lo_hi[:, 0], lo_hi[:, 1], alpha=0.25)
    ax.plot(range(13), s.mean(("chain", "draw")), marker=marker, label=f"{cohort}: projected")
    ax.scatter(retention["year"], retention[cohort], marker="x", color="black", zorder=3)
ax.axvline(7.5, linestyle="--", color="grey")
ax.text(7.7, 900, "projection", color="grey")
ax.set_xlabel("Year")
ax.set_ylabel("Customers still active (of 1,000)")
ax.set_title("sBG projection, posterior mean and 94% HDI; x = actual")
ax.legend()
plt.tight_layout()
plt.show()

rate_t = survival.sel(cohort="regular") / survival.sel(cohort="regular").shift(year=1)
print("Regular, retention rate in year t (posterior mean):",
      ", ".join(f"{t}: {float(rate_t.sel(year=t).mean()):.3f}" for t in range(1, 13)))

**Explain.** Compare with your prediction for the retention rate after year 7. Every customer's
churn probability θ is constant, so why does the retention rate of the whole cohort keep rising?

<details><summary>Why this solution works</summary>

Customers differ: θ ~ Beta(α, β). Those with a high θ leave early, so each year the survivors
are a more loyal group, and the cohort's retention rate rises even though no individual's
probability changes. That is why a model fitted to years 0 to 7 projects years 8 to 12 well,
while a curve that assumes a constant retention rate underestimates later survivors. The
encoding matters: in `ShiftedBetaGeoModel` the likelihood is `pm.Censored(..., upper=T)`, so a
survivor must be censored one period after the last observed renewal (`recency = T = n + 1`).
With `T = n`, survivors look like customers who may have left in year n, and Regular's α and β
come out as 0.718 and 1.208 instead of 0.704 and 1.182.
</details>

## Decision · What is a customer worth?

The CFO asks what the customer base is worth. The cell below gives the number, with its 94% HDI,
for 12 and 36 months at 0.8% a month, and the assumption from the Exercise 4 grid that moves it
most.

In [ ]:
def summarize(x):
    lo, hi = checks.interval(x, INTERVAL, "hdi")
    return float(x.mean()), lo, hi


n_customers = len(rfm)
value = {}
for months in (12, 36):
    per_customer = clv_fn(months, MONTHLY_RATE)
    value[months] = {"total": summarize(per_customer.sum("customer_id")),
                     "average": summarize(per_customer.mean("customer_id"))}


def at(months, rate):
    row = grid[(grid["horizon_months"] == months) & np.isclose(grid["monthly_rate"], rate)]
    return float(row["total_mean"].iloc[0])


horizon_move = at(36, MONTHLY_RATE) - at(12, MONTHLY_RATE)
rate_move = at(36, 0.02) - at(36, 0.0)
bigger = "the horizon" if abs(horizon_move) > abs(rate_move) else "the discount rate"
mode = "QUICK run: treat this as a rough answer." if QUICK else "FULL run."
t12, t36 = value[12]["total"], value[36]["total"]
a12, a36 = value[12]["average"], value[36]["average"]
print(f"Number          : {n_customers:,} customers are worth ${t12[0]:,.0f} in revenue over 12 months"
      f" (94% HDI ${t12[1]:,.0f} to ${t12[2]:,.0f})")
print(f"                  and ${t36[0]:,.0f} over 36 months (94% HDI ${t36[1]:,.0f} to"
      f" ${t36[2]:,.0f}), at 0.8% a month.")
print(f"                  Average customer: ${a12[0]:.0f} (12 months, ${a12[1]:.0f} to ${a12[2]:.0f}),"
      f" ${a36[0]:.0f} (36 months, ${a36[1]:.0f} to ${a36[2]:.0f}).")
print(f"                  At Module 5's assumed 30% gross margin, the 12-month average is"
      f" ${0.3 * a12[0]:.0f} of margin.")
print(f"Largest move    : 12 -> 36 months at 0.8% a month: {horizon_move:+,.0f};"
      f" 0% -> 2% a month at 36 months: {rate_move:+,.0f}. {bigger.capitalize()} moves it most.")
print(f"                  Spend model: frequency-spend correlation {corr_retailer:.2f} (Exercise 1).")
print("Rule            : quote CLV as an expected value with its interval, horizon and monthly rate;")
print("                  when two assumptions are uncertain, report the one that moves the total most.")
print("Recommendation  : (yours) the number for the CFO, the assumption that moves it most, and what")
print("                  would change it (a margin, a horizon, a spend model).")
print("Your sentence   : ________________________________________________")
print(f"Mode            : {mode}")

**Decide.** Write your sentence for the CFO in the cell below: the number with its interval,
horizon and rate, the assumption that moves it most, and what would change it.

## Stretch (optional) · The model that made the data

The retailer's purchases come from a Pareto/NBD process with acquisition-channel effects. Fit
`ParetoNBDModel` by MAP (a point estimate, seconds) with the channel indicators as covariates on
both purchase and dropout, and compare its 12-month CLV with the BG/NBD's and the truth, in total
and by channel. This is the bridge to Module 5, which values customers by channel.

In [ ]:
covariates = ["channel_social", "channel_referral"]
rfm_cov = rfm[["customer_id", "frequency", "recency", "T", "monetary_value", *covariates]]
pnbd, pnbd_estimates = recovery.fit_pnbd_map(rfm_cov, covariate_cols=covariates)
clv_pnbd = gg_clv.expected_customer_lifetime_value(
    transaction_model=pnbd, data=rfm_cov, future_t=12, discount_rate=MONTHLY_RATE, time_unit="W"
)
existing = truth["value_by_channel"]["existing_customers_at_cal_end"]
channel = rfm["acquisition_channel"].to_numpy()
rows = []
for c in ("search", "social", "referral", "all"):
    mask = np.ones(len(rfm), bool) if c == "all" else channel == c
    rows.append({
        "channel": c,
        "Pareto/NBD (MAP)": float(clv_pnbd.isel(chain=0, draw=0).values[mask].sum()),
        "BG/NBD (posterior mean)": float(clv_lib.mean(("chain", "draw")).values[mask].sum()),
        "truth": truth_total if c == "all" else existing[c]["total_discounted_clv"],
    })
pd.DataFrame(rows).set_index("channel").round(0)

In [ ]:
# Generated by scripts/gen_notebooks.py from _variables.yml: do not edit this cell.
# Which checkpoints passed, and on whose code; then the run record (also saved as run_record.json).
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py from _variables.yml. Do not edit this cell. -->
---
**Next:** [Module 5 · From CLV to decisions](https://colab.research.google.com/github/project-delphi/marketing-statistics-workshop/blob/main/labs/python/05-clv-decisions.ipynb) (Python)

[Workshop site](https://project-delphi.github.io/marketing-statistics-workshop/) · [All notebooks](https://project-delphi.github.io/marketing-statistics-workshop/notebooks.html) · [Source](https://github.com/project-delphi/marketing-statistics-workshop)

Genial Labs · text CC BY 4.0, code MIT